# 🎮 VideoGame Ranking in a notebook

This notebook shows, one step at a time, how to use the `vgrank` package to rate
video games and build a personal ranking:

1. **Setup**: import the package and prepare the data
2. **Load the collection** of rated games from a CSV file
3. **The ranking**: leaderboard, podium and filters
4. **Add a game**: search it on the RAWG API, rate it and save it
5. **Edit and remove** a game
6. **Insights**: statistics and interactive charts
7. **Export** the ranking to a CSV file

Every function used here comes from the `vgrank/` folder.

**How to run it.** Install the requirements (`pip install -r requirements.txt`),
then start Jupyter from the project folder with `jupyter lab` and open
`notebooks/vgrank_walkthrough.ipynb`. Run the cells from top to bottom
(*Run → Run All Cells*).

## 1. Setup

The notebook lives in the `notebooks/` folder, while the `vgrank` package and the
`data/` folder are one level up, in the project folder. Python only looks for
packages in a few places, so we add the project folder to that list (`sys.path`).

In [1]:
import os
import shutil
import sys

import pandas as pd
from IPython.display import Image, display

# Jupyter runs the notebook from the folder that contains it (notebooks/),
# so the project folder is its parent.
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)

# Let Python find the vgrank package, which lives in the project folder.
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("Project folder:", os.path.basename(PROJECT_ROOT))

Project folder: vgrank


Now we can import the modules of the package we need.

In [2]:
from vgrank import analysis, plots
from vgrank.api import ApiError, search_games
from vgrank.collection import GameCollection
from vgrank.models import Game
from vgrank.scoring import CATEGORIES, WEIGHTS, overall_score

### Working on a copy of the data

The collection saves every change to its CSV file. In this notebook we are going to
add, edit and remove games just to see how it works, so we make a **copy** of the
file and work on the copy. The original file is never touched, and running the
notebook again always starts from the same data.

> If you want the notebook to change your real ranking, use
> `CSV_PATH = ORIGINAL_CSV` instead.

In [3]:
ORIGINAL_CSV = os.path.join(PROJECT_ROOT, "data", "games.csv")
CSV_PATH = os.path.join(NOTEBOOK_DIR, "games_copy.csv")

# shutil.copy overwrites the copy if it already exists.
shutil.copy(ORIGINAL_CSV, CSV_PATH)
# relpath shows the path starting from the project folder, which is shorter to read.
print("Working on:", os.path.relpath(CSV_PATH, PROJECT_ROOT))

Working on: notebooks/games_copy.csv


## 2. Load the collection

A `GameCollection` reads the CSV file and turns every row into a `Game` object.
`load()` returns the collection itself, so creating and loading can be written on
one line.

In [4]:
collection = GameCollection(csv_path=CSV_PATH).load()

print("Games in the collection:", len(collection))

# collection.games is a normal Python list of Game objects.
collection.games[:3]

Games in the collection: 16


[Game('The Witcher 3: Wild Hunt', overall=8.9),
 Game('Grand Theft Auto V', overall=8.1),
 Game('Portal 2', overall=8.2)]

Each `Game` stores the information that comes from RAWG (title, genres, ...) and
the scores given by the user. It also has a few methods, for example `overall()`,
which computes the weighted score.

In [5]:
first_game = collection.games[0]

print("Title:        ", first_game.title)
print("Release year: ", first_game.release_year())
print("Genres:       ", first_game.genres)  # a list of strings
print("Scores:       ", first_game.scores)  # a dictionary category -> score
print("Overall score:", first_game.overall())

Title:         The Witcher 3: Wild Hunt
Release year:  2015
Genres:        ['Action', 'RPG']
Scores:        {'gameplay': 9, 'story': 10, 'graphics': 9, 'sound': 10, 'replayability': 7}
Overall score: 8.9


## 3. The ranking

The overall score is a weighted average of the five categories. The weights are
defined once in `vgrank/scoring.py`:

In [6]:
for category in CATEGORIES:
    print(f"{category:<15} {WEIGHTS[category]:.0%}")

gameplay        30%
story           20%
graphics        20%
sound           10%
replayability   20%


`collection.ranking()` returns a pandas DataFrame with two extra columns:
`overall` (the weighted score) and `rank` (1 = best game).

In [7]:
ranking = collection.ranking()

# Show only the most interesting columns.
ranking[["rank", "title", "genres", "overall"] + CATEGORIES]

,rank,title,genres,overall,gameplay,story,graphics,sound,replayability
0,1,Hades,"Action, RPG, Indie",9.3,10,8,9,9,10
1,2,Hollow Knight,"Platformer, Indie, Action",9.0,10,8,9,10,8
2,3,The Witcher 3: Wild Hunt,"Action, RPG",8.9,9,10,9,10,7
3,4,Red Dead Redemption 2,"Action, Adventure",8.6,8,10,10,10,6
4,5,Portal 2,"Shooter, Puzzle",8.2,10,9,7,8,6
5,6,Elden Ring,"Action, RPG",8.1,9,6,9,8,8
6,7,Grand Theft Auto V,"Action, Adventure",8.1,8,8,8,9,8
7,8,Stardew Valley,"Indie, RPG, Simulation",8.1,9,6,7,8,10
8,9,The Elder Scrolls V: Skyrim,"Action, RPG",7.9,8,7,7,9,9
9,10,Ori and the Blind Forest,"Adventure, Platformer",7.8,8,7,10,10,5


### The podium

The first three games get a medal. Their cover images are shown with
`IPython.display`, which can display pictures inside a notebook.

In [8]:
MEDALS = ["🥇", "🥈", "🥉"]

podium = ranking.head(3)

for _, row in podium.iterrows():
    medal = MEDALS[row["rank"] - 1]
    print(f"{medal} {row['title']} - overall {row['overall']:.2f}")
    if row["image_url"] != "":
        display(Image(url=row["image_url"], width=300))

🥇 Hades - overall 9.30


🥈 Hollow Knight - overall 9.00


🥉 The Witcher 3: Wild Hunt - overall 8.90


### Filters

We can keep only some games: those of certain genres and with a minimum overall
score. The filters are plain variables: change them and run the cell again.

The `genres` column is a text like `"Action, RPG"`, so a small function checks
whether a game has at least one of the wanted genres.

In [9]:
def has_any_genre(text, wanted_genres):
    """Return True if the comma separated text contains one of wanted_genres."""
    for genre in text.split(", "):
        if genre in wanted_genres:
            return True
    return False


# Try to change these two values.
selected_genres = ["RPG", "Indie"]  # use an empty list [] to keep every genre
min_overall = 8.0

filtered = ranking[ranking["overall"] >= min_overall]
if len(selected_genres) > 0:
    keep = filtered["genres"].apply(has_any_genre, wanted_genres=selected_genres)
    filtered = filtered[keep]

print(f"{len(filtered)} games · average overall score {filtered['overall'].mean():.2f}")
filtered[["rank", "title", "genres", "overall"]]

5 games · average overall score 8.68


,rank,title,genres,overall
0,1,Hades,"Action, RPG, Indie",9.3
1,2,Hollow Knight,"Platformer, Indie, Action",9.0
2,3,The Witcher 3: Wild Hunt,"Action, RPG",8.9
5,6,Elden Ring,"Action, RPG",8.1
7,8,Stardew Valley,"Indie, RPG, Simulation",8.1


## 4. Add a game

### Search on RAWG

`search_games` calls the [RAWG API](https://rawg.io/apidocs) and returns a list
of dictionaries, one per game. It needs the API key written in the `.env` file
(see the README).

If the key is missing or the internet connection does not work, the function
raises an `ApiError`. We catch it with `try` / `except`, so the notebook keeps
working even without a key.

In [ ]:
query = "Celeste"

try:
    results = search_games(query, max_results=5)
except ApiError as error:
    print("Search failed:", error)
    results = []

# Show the release year next to the title, so remakes and sequels can be told apart.
for position, game in enumerate(results):
    year = game["released"][:4] if game["released"] else "unknown year"
    print(f"{position}: {game['title']} ({year})")

0: Celeste (2018)
1: Celeste 2.7 (2020)
2: Celeste's Mathland (2018)
3: Celeste's Math Classroom (2021)
4: Celeste - Break the Targets (2021)


### Pick the right result

A search often returns several games with similar names, so we choose the right
one by its position in the list. If the search did not work, we write the game information
by hand, in the same format returned by `search_games`.

In [11]:
if len(results) > 0:
    chosen = results[0]  # change the position if the first result is not the right game
else:
    chosen = {
        "game_id": 22121,
        "title": "Celeste",
        "released": "2018-01-25",
        "genres": ["Platformer", "Indie"],
        "platforms": ["PC", "PlayStation 4", "Xbox One", "Nintendo Switch"],
        "metacritic": 91,
        "image_url": "",
    }

chosen

{'game_id': 22121,
 'title': 'Celeste',
 'released': '2018-01-25',
 'genres': ['Platformer', 'Indie'],
 'platforms': ['PC',
  'Xbox One',
  'PlayStation 4',
  'Nintendo Switch',
  'macOS',
  'Linux'],
 'metacritic': 91,
 'image_url': 'https://media.rawg.io/media/games/594/59487800889ebac294c7c2c070d02356.jpg'}

### Rate it

The scores are a dictionary with one entry per category. Every score must be
between 1 and 10.

In [12]:
scores = {
    "gameplay": 9,
    "story": 7,
    "graphics": 8,
    "sound": 9,
    "replayability": 7,
}

print(f"Overall score: {overall_score(scores):.2f}")

Overall score: 8.00


### Save it

We build a `Game` object and add it to the collection. `add()` also saves the CSV
file. If the game is already in the collection, `add()` raises a `ValueError`.

In [13]:
new_game = Game(
    game_id=chosen["game_id"],
    title=chosen["title"],
    released=chosen["released"],
    genres=chosen["genres"],
    platforms=chosen["platforms"],
    metacritic=chosen["metacritic"],
    image_url=chosen["image_url"],
    scores=scores,
)

try:
    collection.add(new_game)
    print(f"'{new_game.title}' added with an overall score of {new_game.overall():.2f}")
except ValueError as error:
    print("Not added:", error)

'Celeste' added with an overall score of 8.00


The ranking is not updated automatically: we have to build it again. Then we
can look for the new game with a boolean filter.

In [14]:
ranking = collection.ranking()
ranking[ranking["title"] == new_game.title][["rank", "title", "overall"]]

,rank,title,overall
8,9,Celeste,8.0


## 5. Edit and remove a game

The collection has two methods to change the games it contains:
`update_scores()` and `remove()`. Both need the `game_id` of the game.

In [15]:
new_scores = {
    "gameplay": 10,
    "story": 8,
    "graphics": 8,
    "sound": 10,
    "replayability": 8,
}

collection.update_scores(new_game.game_id, new_scores)

ranking = collection.ranking()
ranking[ranking["title"] == new_game.title][["rank", "title", "overall"]]

,rank,title,overall
3,4,Celeste,8.8


Scores are checked before being saved: a score outside the 1-10 range raises a
`ValueError` and nothing is changed.

In [16]:
wrong_scores = dict(new_scores)  # a copy of the dictionary
wrong_scores["story"] = 11

try:
    collection.update_scores(new_game.game_id, wrong_scores)
except ValueError as error:
    print("Not saved:", error)

Not saved: Score for 'story' must be between 1 and 10, got 11


To remove a game we first need its `game_id`. Here we look it up from the title
in the ranking table.

In [17]:
title_to_remove = "Minecraft"

matches = ranking[ranking["title"] == title_to_remove]
if len(matches) == 0:
    print(f"'{title_to_remove}' is not in the collection")
else:
    game_id = int(matches["game_id"].iloc[0])
    collection.remove(game_id)
    print(f"'{title_to_remove}' removed. Games left: {len(collection)}")

'Minecraft' removed. Games left: 16


## 6. Insights

The functions in `vgrank/analysis.py` compute statistics with pandas, numpy and
scipy; the functions in `vgrank/plots.py` turn them into interactive Plotly
charts. Every plot function returns a figure, which we display with `fig.show()`.

First, we rebuild the ranking so that it includes the changes of the previous
sections.

In [18]:
ranking = collection.ranking()
print("Games in the ranking:", len(ranking))

Games in the ranking: 16


### Categories

How are the scores spread in each category? Hover over a dot to see the game.

In [19]:
fig = plots.category_boxplot(ranking)
fig.show()

The same information as numbers: mean, standard deviation, minimum and maximum
of every category.

In [20]:
analysis.category_summary(ranking)

,mean,std,min,max
gameplay,8.56,1.09,7.0,10.0
story,7.38,1.75,4.0,10.0
graphics,8.19,1.17,6.0,10.0
sound,8.94,1.00,7.0,10.0
replayability,7.25,1.65,5.0,10.0


Are the categories related? A value close to 1 means that games with a high score
in one category also tend to have a high score in the other.

In [21]:
correlation = analysis.category_correlation(ranking)
fig = plots.correlation_heatmap(correlation)
fig.show()

Where does each game stand out? Scores are standardised with a z-score first, so
that a category rated high for every game does not always win.

In [22]:
analysis.strongest_category(ranking)

,title,strongest,z_score
0,Hades,replayability,1.72
1,Hollow Knight,gameplay,1.36
2,The Witcher 3: Wild Hunt,story,1.55
3,Celeste,gameplay,1.36
4,Red Dead Redemption 2,graphics,1.60
5,Portal 2,gameplay,1.36
6,Elden Ring,graphics,0.72
7,Grand Theft Auto V,replayability,0.47
8,Stardew Valley,replayability,1.72
9,The Elder Scrolls V: Skyrim,replayability,1.09


### Genres and years

Average overall score per genre. `min_games` hides the genres with too few games,
where the average is not very meaningful.

In [23]:
genre_table = analysis.score_by_genre(ranking, min_games=2)

if len(genre_table) == 0:
    print("No genre has that many games.")
else:
    fig = plots.genre_bar_chart(genre_table)
    fig.show()

In [24]:
counts = analysis.games_per_year(ranking)
fig = plots.games_per_year_chart(counts)
fig.show()

### My scores vs the critics

Metacritic scores go from 0 to 100, so they are divided by 10 to be compared
with ours. Spearman's correlation tells how similar the two *orderings* are:
1 means we rank the games exactly like the critics do.

In [25]:
comparison, summary = analysis.compare_with_metacritic(ranking)

print("Games with a Metacritic score:", summary["n_games"])
print("Spearman correlation:        ", summary["correlation"])
print("p-value:                     ", summary["p_value"])

Games with a Metacritic score: 16
Spearman correlation:         0.423
p-value:                      0.102


Points on the dashed diagonal are games where we agree with the critics. Points
above it are games we like more than the critics, points below are games we
like less.

In [26]:
if summary["correlation"] is None:
    print("At least 3 games with a Metacritic score are needed for this chart.")
else:
    fig = plots.metacritic_scatter(comparison)
    fig.show()

The five biggest disagreements. We sort by the **absolute** difference, so that
large negative differences count as much as large positive ones.

In [27]:
order = comparison["difference"].abs().sort_values(ascending=False).index
comparison.loc[order].head(5)[["title", "overall", "metacritic_10", "difference"]]

,title,overall,metacritic_10,difference
14,Grand Theft Auto IV,6.9,9.5,-2.6
6,Elden Ring,8.1,9.6,-1.5
9,The Elder Scrolls V: Skyrim,7.9,9.4,-1.5
5,Portal 2,8.2,9.5,-1.3
12,Cyberpunk 2077,7.4,8.6,-1.2


## 7. Export the ranking

We save the ranking to a new CSV file with `to_csv` and, to check it, read it
back with `read_csv`.

In [28]:
export_path = os.path.join(NOTEBOOK_DIR, "my_ranking.csv")
ranking.to_csv(export_path, index=False)

exported = pd.read_csv(export_path)
print(f"Saved {len(exported)} games to {os.path.relpath(export_path, PROJECT_ROOT)}")
exported[["rank", "title", "overall"]].head()

Saved 16 games to notebooks/my_ranking.csv


,rank,title,overall
0,1,Hades,9.3
1,2,Hollow Knight,9.0
2,3,The Witcher 3: Wild Hunt,8.9
3,4,Celeste,8.8
4,5,Red Dead Redemption 2,8.6
